In [2]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
import seaborn as sns
from sklearn.metrics import silhouette_score
import numpy as np
from scipy.stats import chi2

import warnings
warnings.filterwarnings("ignore")
import glob


In [3]:

files = glob.glob('/data1st2/junyi/output/atac1112/subset/region_nt/*sc_subset.h5ad')
for file in files:
    if os.path.exists(f'/data1st2/junyi/output/atac1112/subset/region_nt/{os.path.basename(file).replace("_sc_subset.h5ad", "_sc_bulk.h5ad")}'):
        continue
    # adata = sc.read_h5ad(file)
    # sc.pp.calculate_qc_metrics(adata, inplace=True)
    # sc.pl.umap(adata, color=['total_counts', 'n_genes_by_counts','sample'], wspace=0.4)
    # sc.pl.violin(adata, ['total_counts', 'n_genes_by_counts'],jitter=False, groupby='sample', rotation=90)    
    adata = sc.read_h5ad(file)
    ad_bulk = sc.get.aggregate(adata,by='celltype.L2.condition',func='sum',layer="counts")
    ad_bulk.X = ad_bulk.layers['sum'].astype(int)
    sc.pp.normalize_total(ad_bulk, target_sum=1e6)
    sc.pp.log1p(ad_bulk)
    ad_bulk.write_h5ad(f'/data1st2/junyi/output/atac1112/subset/region_nt/{os.path.basename(file).replace("_sc_subset.h5ad", "_sc_bulk.h5ad")}')

In [4]:
df_ccre=pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/cCRE_annotated.csv')

In [10]:
df_ccre

,chr,start,end,names,gene_name,gene_id,gstart,gend,strand,annotation_x,distance,primary_region,secondary_region,encodeCCRE,Sex,Ensemble,Gene
0,chr1,3003518,3004019,chr1:3003518-3004019,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,69234,distal,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3003518-3004019
1,chr1,3007481,3007982,chr1:3007481-3007982,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,65271,distal,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3007481-3007982
2,chr1,3012480,3012981,chr1:3012480-3012981,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,60272,distal,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3012480-3012981
3,chr1,3013424,3013925,chr1:3013424-3013925,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,59328,distal,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3013424-3013925
4,chr1,3014717,3015218,chr1:3014717-3015218,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,58035,distal,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3014717-3015218
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667327,chrY,90812660,90813161,chrY:90812660-90813161,Gm47283,ENSMUSG00000096768.8,90784737,90816465,+,genebody,0,genebody,NaN,NaN,M,ENSMUSG00000096768.8,chrY:90812660-90813161
1667328,chrY,90811433,90811934,chrY:90811433-90811934,Gm47283,ENSMUSG00000096768.8,90784737,90816465,+,genebody,0,genebody,NaN,NaN,M,ENSMUSG00000096768.8,chrY:90811433-90811934
1667329,chrY,90813596,90814097,chrY:90813596-90814097,Gm47283,ENSMUSG00000096768.8,90784737,90816465,+,genebody,0,genebody,NaN,NaN,M,ENSMUSG00000096768.8,chrY:90813596-90814097
1667330,chrY,90825201,90825702,chrY:90825201-90825702,Gm47283,ENSMUSG00000096768.8,90784737,90816465,+,genebody,-8737,downstream,NaN,NaN,M,ENSMUSG00000096768.8,chrY:90825201-90825702


In [21]:
cicre_files = glob.glob('/data2st1/junyi/output/atac1112/cicre/*_ALL_ccans.csv')

for file in cicre_files:
    celltype = os.path.basename(file).replace('_ALL_ccans.csv','')
    df_cicre = pd.read_csv(file,index_col=0)
    df_cicre['names'] = df_cicre['Peak'].str.split('_').apply(lambda x: x[0] + ':' + x[1] + '-' + x[2])
    df_cicre_merged = df_cicre.merge(df_ccre, left_on='names', right_on='names', how='left')
    df_cicre_merged['CCAN_annotation'] = celltype+"_"+df_cicre_merged['CCAN'].astype(str)
    df_cicre_merged.to_csv(file.replace('_ALL_ccans.csv','_ALL_ccans_annotated.csv'), index=False)

In [53]:
df_deg = pd.read_csv("/data2st1/junyi/output/atac1112/dar/celltype.L2/mast_ngsa_noco_degs_fdr_log2fc0_filtered.csv",index_col=0)
df_deg['Region Subclass'] = df_deg['Region subclass'].str.replace(" ",'_')
df_deg['Region Subclass'] = df_deg['Region Subclass'].str.replace("/",'-')
df_deg = df_deg[df_deg.Sex=='M']
df_deg = df_deg[df_deg.Region.isin(['HPF','PFC','AMY'])]

In [97]:
adata_atac = sc.read_h5ad('/data1st2/junyi/output/atac1112/subset/region_nt/HIP_HIP_Glut.h5ad')

In [77]:
df_gen_function

,Channel family (expanded subfamily),Category
Gene,,
Scn1a,Voltage-gated sodium channel — Nav1.1 α,Voltage-gated sodium channel
Scn2a,Voltage-gated sodium channel — Nav1.2 α,Voltage-gated sodium channel
Scn3a,Voltage-gated sodium channel — Nav1.3 α,Voltage-gated sodium channel
Scn4a,Voltage-gated sodium channel — Nav1.4 α,Voltage-gated sodium channel
Scn5a,Voltage-gated sodium channel — Nav1.5 α,Voltage-gated sodium channel
...,...,...
Glra1,配体门控甘氨酸受体,Gly
Glra2,配体门控甘氨酸受体,Gly
Glra3,配体门控甘氨酸受体,Gly


In [82]:
ID_GEN = adata_atac.var['gene_name'].isin(df_gen_function.index)

In [99]:
adata_focus = adata_atac[:,ID_GEN]

In [112]:
sc.pp.filter_genes(adata_focus, min_cells=10)

In [118]:
adata_focus.obs

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,celltype.L1_ct,Sample_name,Condition,Region,celltype.L2.raw,region_nt,celltype.L3,celltype.L4,celltype.L2.refined,expriment
MC39C_HIP:AAACGAAAGATGAGGA-1,MC39C_HIP,0.172442,0.002326,15,14,0,2,2,2,2,...,Glut,MC39C_HIP,MC,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-1,HPF_DG_GC_Glut-1-0,HPF DG GC Glut,MC
MC39C_HIP:AAACGAAAGGCTTAAA-1,MC39C_HIP,0.071620,0.063665,0,2,2,3,3,3,3,...,Glut,MC39C_HIP,MC,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-0,HPF_DG_GC_Glut-0-0,HPF DG GC Glut,MC
MC39C_HIP:AAACGAAAGGGACGTT-1,MC39C_HIP,0.118938,0.018022,0,2,2,3,3,3,3,...,Glut,MC39C_HIP,MC,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-0,HPF_DG_GC_Glut-0-1,HPF DG GC Glut,MC
MC39C_HIP:AAACGAAAGGTTCGTT-1,MC39C_HIP,0.114256,0.019980,7,3,0,2,2,2,2,...,Glut,MC39C_HIP,MC,HIP,HPF CA1 Glut,HIP_Glut,HPF_CA1_Glut-1,HPF_CA1_Glut-1-0,HPF CA1 Glut,MC
MC39C_HIP:AAACGAAAGTCAACTC-1,MC39C_HIP,0.087126,0.036101,0,2,2,3,3,3,3,...,Glut,MC39C_HIP,MC,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-0,HPF_DG_GC_Glut-0-1,HPF DG GC Glut,MC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MW51A_HIP:TTTGTGTTCAAACCAC-1,MW51A_HIP,0.102530,0.008243,15,14,0,0,1,1,1,...,Glut,MW51A_HIP,MW,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-1,HPF_DG_GC_Glut-1-0,HPF DG GC Glut,MW
MW51A_HIP:TTTGTGTTCAGTCACA-1,MW51A_HIP,0.067125,0.099081,0,2,2,3,3,3,3,...,Glut,MW51A_HIP,MW,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-0,HPF_DG_GC_Glut-0-1,HPF DG GC Glut,MW
MW51A_HIP:TTTGTGTTCATTCTTG-1,MW51A_HIP,0.063649,0.042679,0,2,2,3,3,3,3,...,Glut,MW51A_HIP,MW,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-0,HPF_DG_GC_Glut-0-0,HPF DG GC Glut,MW
MW51A_HIP:TTTGTGTTCCGTTTCG-1,MW51A_HIP,0.067596,0.036308,13,3,0,2,2,2,2,...,Glut,MW51A_HIP,MW,HIP,HPF DG GC Glut,HIP_Glut,HPF_DG_GC_Glut-1,HPF_DG_GC_Glut-1-0,HPF DG GC Glut,MW


In [121]:
adata_focunt_psbulk = sc.get.aggregate(adata_focus,by='celltype.L2.Condition',func='sum',layer="count")

In [ ]:
# Calculate the fold change between conditions
adata_focunt_psbulk.obs['condition'] = adata_focunt_psbulk.obs['celltype.L2.Condition'].str.split('_').str[-1]

In [ ]:
# calculate fold change


NotImplementedError: 

In [109]:
adata_focus.layers['count'].sum(axis=1)

matrix([[ 59],
        [591],
        [216],
        ...,
        [384],
        [967],
        [162]], dtype=uint64)

In [104]:
adata_focus.layers['count']

<Compressed Sparse Row sparse matrix of dtype 'uint32'
	with 16411674 stored elements and shape (41311, 32215)>

In [61]:
df_ccre.index = df_ccre['names']

In [68]:
len(set(adata_atac.var.index))

1660766

In [70]:
df_ccre.drop_duplicates(subset=['names'], inplace=True)

In [74]:
adata_atac.var = df_ccre.loc[adata_atac.var.index]

In [ ]:
adata_atac.var['gene_names'] = adata_atac.var_names

AnnData object with n_obs × n_vars = 2141 × 1660766
    obs: 'sample', 'doublet_probability', 'doublet_score', 'leiden', 'leiden_default', 'leiden_res_0.1', 'leiden_res_0.2', 'leiden_res_0.3', 'leiden_res_0.4', 'leiden_res_0.5', 'leiden_res_0.6', 'leiden_res_0.7', 'leiden_res_0.8', 'leiden_res_0.9', 'leiden_res_1.0', 'leiden_res_1.1', 'leiden_res_1.2', 'leiden_res_1.3', 'leiden_res_1.4', 'leiden_res_1.5', 'leiden_res_1.6', 'leiden_res_1.7', 'leiden_res_1.8', 'leiden_res_1.9', 'celltype.L2.Condition', 'celltype.L1', 'celltype.L2', 'Neurotransmitter_celltype', 'celltype.L1_ct', 'Sample_name', 'Condition', 'Region', 'celltype.L2.raw', 'region_nt', 'celltype.L3', 'celltype.L4', 'celltype.L2.refined', 'expriment'
    var: 'chr', 'start', 'end', 'names', 'gene_name', 'gene_id', 'gstart', 'gend', 'strand', 'annotation_x', 'distance', 'primary_region', 'secondary_region', 'encodeCCRE', 'Sex', 'Ensemble', 'Gene'
    uns: 'log1p'
    obsm: 'X_spectral', 'X_umap'
    layers: 'count'

In [47]:
df_nat = pd.read_csv('/data2st2/junyi/output/atac1112/tobiasbam/HIPcisbp_BIND/HIP_HPF_DG_GC_Glut_MC_footprints.bw/bindetect_distances.txt', sep='\t')

In [48]:
df_nat.index = df_nat.columns

In [42]:
df_result = pd.read_excel('/data2st2/junyi/output/atac1112/tobiasbam/HIPcisbp_BIND/HIP_HPF_Subiculum_IT_Glut_MC_footprints.bw/bindetect_results.xlsx', index_col=0)

In [15]:
df_cicre.merge(df_ccre, left_on='names', right_on='names', how='left')

,Peak,CCAN,names,chr,start,end,gene_name,gene_id,gstart,gend,strand,annotation_x,distance,primary_region,secondary_region,encodeCCRE,Sex,Ensemble,Gene
0,chr19_5711163_5711664,8,chr19:5711163-5711664,chr19,5711163,5711664,Ehbp1l1,ENSMUSG00000024937.15,5707375,5726317,-,genebody,0,genebody,NaN,NaN,M,ENSMUSG00000024937.15,chr19:5711163-5711664
1,chr19_5567437_5567938,8,chr19:5567437-5567938,chr19,5567437,5567938,Ap5b1,ENSMUSG00000049562.6,5568024,5571261,+,genebody,87,promoter,cpg,"dELS,CTCF-bound",M,ENSMUSG00000049562.6,chr19:5567437-5567938
2,chr19_5844400_5844901,8,chr19:5844400-5844901,chr19,5844400,5844901,Neat1,ENSMUSG00000092274.3,5824707,5845478,-,genebody,0,genebody,NaN,NaN,M,ENSMUSG00000092274.3,chr19:5844400-5844901
3,chr19_5835489_5835990,8,chr19:5835489-5835990,chr19,5835489,5835990,Neat1,ENSMUSG00000092274.3,5824707,5845478,-,genebody,0,genebody,NaN,NaN,M,ENSMUSG00000092274.3,chr19:5835489-5835990
4,chr19_5844920_5845421,8,chr19:5844920-5845421,chr19,5844920,5845421,Neat1,ENSMUSG00000092274.3,5824707,5845478,-,genebody,0,exon,NaN,pELS,M,ENSMUSG00000092274.3,chr19:5844920-5845421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
379,chr17_35953560_35954061,531,chr17:35953560-35954061,chr17,35953560,35954061,Gm8801,ENSMUSG00000092284.1,35947245,35952297,+,genebody,-1264,downstream,NaN,NaN,M,ENSMUSG00000092284.1,chr17:35953560-35954061
380,chr17_35274688_35275189,531,chr17:35274688-35275189,chr17,35274688,35275189,Gm18733,ENSMUSG00000092444.1,35278175,35279414,-,genebody,2987,downstream,repeat,"CTCF-only,CTCF-bound",M,ENSMUSG00000092444.1,chr17:35274688-35275189
381,chr11_100619934_100620435,594,chr11:100619934-100620435,chr11,100619934,100620435,Dnajc7,ENSMUSG00000014195.15,100582817,100620168,-,genebody,0,promoter,NaN,"dELS,CTCF-bound",M,ENSMUSG00000014195.15,chr11:100619934-100620435
382,chr11_100622638_100623139,594,chr11:100622638-100623139,chr11,100622638,100623139,Nkiras2,ENSMUSG00000017837.12,100619243,100627607,+,genebody,0,promoter,cpg,"dELS,CTCF-bound",M,ENSMUSG00000017837.12,chr11:100622638-100623139


In [4]:
adata = sc.read_h5ad(files[1])


In [1]:
adata.var

NameError: name 'adata' is not defined

In [ ]:
for mod in ['5mC', '5hmC']:
    df_dmr_5mc = df_dmr[df_dmr['mod']==mod]
    df_dmr_5mc['Region'] = df_dmr_5mc['comparision'].str[3:6]
    df_dmr_5mc = df_dmr_5mc#[df_dmr_5mc['Region']=='AMY']
    df_dmr_5mc_expanded = df_dmr_5mc.dmr.str.split('[|_:-]', expand=True)
    df_dmr_5mc_expanded.columns = ['methtype', 'motif2', 'chr','start', 'end']
    # Expand DMRs with length <100 to 100bp centered at original center
    df_dmr_5mc_expanded['length'] = df_dmr_5mc_expanded['end'].astype(int) - df_dmr_5mc_expanded['start'].astype(int)
    df_dmr_5mc_expanded['center'] = (df_dmr_5mc_expanded['end'].astype(int) + df_dmr_5mc_expanded['start'].astype(int)) //2
    df_dmr_5mc_expanded['start_expanded'] = df_dmr_5mc_expanded['center'] - 50
    df_dmr_5mc_expanded['end_expanded'] = df_dmr_5mc_expanded['center'] + 50
    # if legthn <100, expand start = start_expanded, end = end_expanded
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start_expanded']
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end_expanded']

    df_dmr_5mc =  pd.concat([df_dmr_5mc, df_dmr_5mc_expanded], axis=1)
    df_dmr_5mc.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{mod}_annotation.csv')
    df_bed= df_dmr_5mc.loc[:,['chr', 'start', 'end']].drop_duplicates()
    df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
    df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/dmr_{mod}.bed', sep='\t', header=False, index=False)


In [ ]:
df_bed.drop_duplicates(['chr', 'start', 'end'])

In [ ]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac0627/doublet_filtered.h5ads/_dataset.h5ads')

In [ ]:
%time hm5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5hmC.bed')
hm5c_mat.write(f"output/atac1112/3REGIONS_5hmc_new.h5ads")

In [ ]:
%time m5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5mC.bed')
m5c_mat.write(f"output/atac1112/3REGIONS_5mc_new.h5ads")
adata_concat.close()

In [ ]:
annodict= {}
for methtype in ['5mC', '5hmC']:
    if methtype == '5mC':
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5mC_annotation.csv',index_col=0)
    else:
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5hmC_annotation.csv',index_col=0)
    for region in ['AMY', 'HIP', 'PFC']:
        df_region = annodict[methtype][annodict[methtype]['Region']==region]
        df_region.loc[df_region['length']<100, 'start'] = df_region.loc[df_region['length']<100, 'start_expanded']
        df_region.loc[df_region['length']<100, 'end'] = df_region.loc[df_region['length']<100, 'end_expanded']
        df_bed = df_region.loc[:,['chr', 'start', 'end']].drop_duplicates()
        df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
        df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{methtype}_{region}.bed', sep='\t', header=False, index=False)

In [ ]:
df_region

In [ ]:
for methtype in ['5mC', '5hmC']:
    methtypeL = methtype.lower()
    adata = sc.read_h5ad(f"output/atac1112/3REGIONS_{methtypeL}_new.h5ads")
    dups = adata.var_names.duplicated()
    # Drop duplicated genes
    dmr_mat = adata[:, ~dups].copy()
    #dmr_mat = dmr_mat.obs[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    dmr_mat = dmr_mat[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    adata_dict={}
    for region in ['AMY', 'HIP', 'PFC']:
        df_anno = annodict[methtype]
        df_region = df_anno[df_anno['Region']==region]
        df_region['dmr_name'] = 'chr'+df_region['chr'].astype(str)+':'+df_region['start'].astype(str)+'-'+df_region['end'].astype(str)
        adata_region = dmr_mat[dmr_mat.obs['Region']==region,]
        df_dmr_ano = df_region.drop_duplicates(subset=['dmr_name'])
        df_dmr_ano.drop('gene',axis=1,inplace=True)
        df_dmr_ano.set_index('dmr_name', inplace=True)
        adata_region = adata_region[:, df_dmr_ano.index]
        adata_region.var= df_dmr_ano.loc[adata_region.var_names, :]
        adata_region.var['chr'] = "chr"+adata_region.var['chr'].astype(str)
        #print(region, df_region.shape[0])
        adata_region.write_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")


In [ ]:
def g_test_row(row):
    row = np.array(row, dtype=float)
    total = row.sum()
    expected = np.repeat(total/len(row), len(row))

    # Avoid log(0)
    row_safe = np.where(row > 0, row, 1e-12)

    G = 2 * np.sum(row_safe * np.log(row_safe / expected))
    pval = 1 - chi2.cdf(G, df=len(row)-1)
    return pval

for methtype in ['5mC', '5hmC']:
    for region in ['AMY', 'HIP', 'PFC']:
        adata_region = sc.read_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")
        adata_region.layers['counts'] = adata_region.X
        adata_region.X = adata_region.layers['counts']
        sc.pp.normalize_total(adata_region, target_sum=1e6)
        #sc.pp.log1p(adata_region)
        group_key = "celltype.L1_ct"  # 替换为 obs 的列名，例如 "cell_type"
        agg_adata = sc.get.aggregate(adata_region,by=group_key,func='mean')
        agg_adata.X = agg_adata.layers['mean']
        # # softmax of over 9 classes
        # from scipy.special import softmax
        X = agg_adata.X.T
        X_norm = X / X.sum(axis=1, keepdims=True)
        X_norm = np.nan_to_num(X_norm, nan=1/9)
        df_xnorm = pd.DataFrame(X_norm, index=agg_adata.var_names, columns=agg_adata.obs_names)
        df_xnorm_safe = df_xnorm.replace(0, 1e-12)
        entropy_values = -np.sum(df_xnorm_safe * np.log(df_xnorm_safe), axis=1)
        # 放回数据框
        df_xnorm['entropy'] = entropy_values
        # pvals = np.array([g_test_row(row) for row in agg_adata.X.T])
        # df_xnorm['pval'] = pvals
        df_xnorm.to_csv(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}_celltype_fraction.csv")
        #
        sc.pp.log1p(adata_region)
        sc.tl.rank_genes_groups(adata_region, groupby=group_key, method="wilcoxon",pts=True)
        for ct in adata_region.obs[group_key].unique():
            df_cts = sc.get.rank_genes_groups_df(adata_region, group=ct,pval_cutoff=0.05)
            df_cts.to_csv(f'/data2st1/junyi/output/atac1112/dar/cts/dmr_wilcoxon/{region}_{methtype}_wilcox_{ct}.csv', index=False)
